# RUKOPYS: handwritten line OCR baseline

This notebook builds a first practical baseline for RUKOPYS. We take annotated `handwritten` line regions, crop them from full pages, and train a CRNN + CTC model to recognize text from each crop.

Important boundary: this is **line-level OCR**, not the full Kaggle pipeline yet. For test pages you also need a layout detector that finds line boxes. Here we focus on the OCR core first, because it is the easiest part to understand and debug.

In [ ]:
import os
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## 1. Load the dataset

`gt_only` is the human-annotated train split without silver labels. Each item is a full page image. Its `regions` field contains line/table/formula annotations with bounding boxes and text.

In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "UkrainianCatholicUniversity/rukopys",
    "gt_only",
    split="train",
)

sample = ds[0]
sample.keys()

In [ ]:
plt.figure(figsize=(6, 8))
plt.imshow(sample["image"])
plt.axis("off")

print("image size:", sample["image"].size)
print("source:", sample["source"])
print("regions:", len(sample["regions"]))
print(sample["regions"][0])

## 2. Inspect one line crop

For OCR we need pairs like `image crop -> text`. We take one handwritten region, crop its bounding box, and keep the corresponding transcription as the target.

In [ ]:
region = next(r for r in sample["regions"] if r["type"] == "handwritten" and r["legibility"] == "legible")
x1, y1, x2, y2 = region["bbox"]
crop = sample["image"].crop((x1, y1, x2, y2))

plt.figure(figsize=(10, 2))
plt.imshow(crop)
plt.axis("off")
print(region["text"])

## 3. Build a line-level OCR dataset

We keep only legible handwritten regions. Text normalization collapses several visually similar punctuation variants into one representation.

For validation we split **pages first**, then extract line crops separately. This avoids leakage where lines from the same page or handwriting style appear in both train and validation.

In [ ]:
def normalize_text(text):
    text = text.strip()
    replacements = {
        "\u00ab": '"',
        "\u00bb": '"',
        "\u201c": '"',
        "\u201d": '"',
        "\u201e": '"',
        "\u2014": "-",
        "\u2013": "-",
        "'": "\u02bc",
        "\u2019": "\u02bc",
        "\u2018": "\u02bc",
        "\u200b": "",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    text = " ".join(text.split())
    return text


IMAGE_HEIGHT = 32
MAX_TEXT_LEN = 180
MIN_CROP_SIDE = 8


def build_ocr_samples(pages):
    samples = []
    skipped = Counter()

    for page_id, page in enumerate(pages):
        img = page["image"]

        for region in page["regions"]:
            if region["type"] != "handwritten":
                skipped["not_handwritten"] += 1
                continue
            if region["legibility"] != "legible":
                skipped["not_legible"] += 1
                continue

            text = normalize_text(region.get("text") or "")
            if not text:
                skipped["empty_text"] += 1
                continue
            if len(text) > MAX_TEXT_LEN:
                skipped["too_long"] += 1
                continue

            x1, y1, x2, y2 = region["bbox"]
            crop = img.crop((x1, y1, x2, y2))
            if crop.width < MIN_CROP_SIDE or crop.height < MIN_CROP_SIDE:
                skipped["tiny_crop"] += 1
                continue

            resized_width = max(1, int(round(crop.width * (IMAGE_HEIGHT / crop.height))))
            if resized_width // 4 < len(text):
                skipped["ctc_too_short"] += 1
                continue

            samples.append({
                "image": crop,
                "text": text,
                "source": page["source"],
                "page_id": page_id,
                "bbox": region["bbox"],
            })

    return samples, skipped


from sklearn.model_selection import train_test_split

page_indices = list(range(len(ds)))
train_idx, val_idx = train_test_split(
    page_indices,
    test_size=0.2,
    random_state=SEED,
    stratify=[ds[i]["source"] for i in page_indices],
)

train_pages = ds.select(train_idx)
val_pages = ds.select(val_idx)

train_samples, train_skipped = build_ocr_samples(train_pages)
val_samples, val_skipped = build_ocr_samples(val_pages)
ocr_samples = train_samples + val_samples

print("train pages:", len(train_pages), "val pages:", len(val_pages))
print("train OCR samples:", len(train_samples), "skipped:", dict(train_skipped))
print("val OCR samples:", len(val_samples), "skipped:", dict(val_skipped))

In [ ]:
texts = [x["text"] for x in ocr_samples]
lengths = [len(t) for t in texts]
widths = [x["image"].size[0] for x in ocr_samples]
heights = [x["image"].size[1] for x in ocr_samples]

print("total OCR samples:", len(ocr_samples))
print("train/val OCR samples:", len(train_samples), len(val_samples))
print("text len min/max/avg:", min(lengths), max(lengths), round(sum(lengths) / len(lengths), 2))
print("crop width min/max:", min(widths), max(widths))
print("crop height min/max:", min(heights), max(heights))
print("sources:", Counter(x["source"] for x in ocr_samples))

## 4. Alphabet and CTC encoding

CTC works with class sequences. Class `0` is the blank token. The model may emit blanks between characters or when it is unsure. During decoding we remove blanks and collapse repeated symbols.

In [ ]:
chars = sorted(set("".join(texts)))
blank_token = "<BLANK>"

idx2char = [blank_token] + chars
char2idx = {ch: i for i, ch in enumerate(idx2char)}
num_classes = len(idx2char)

print("num classes:", num_classes)
print("alphabet preview:", "".join(chars[:120]))

In [ ]:
def encode_text(text):
    return [char2idx[ch] for ch in normalize_text(text)]


def ctc_decode(indices):
    result = []
    prev = None

    for idx in indices:
        idx = int(idx)
        if idx != 0 and idx != prev:
            result.append(idx2char[idx])
        prev = idx

    return "".join(result)


assert ctc_decode([0, 5, 5, 0, 6, 6, 0]) == idx2char[5] + idx2char[6]

## 5. Preprocessing, Dataset, and Collate

Line crops have different widths. In each batch we pad images on the right. Height is fixed, while width stays proportional to the original crop. This matters because text is a horizontal sequence.

In [ ]:
# IMAGE_HEIGHT was set while filtering CTC-compatible samples.

def preprocess_image(image, height=IMAGE_HEIGHT):
    image = image.convert("L")
    w, h = image.size
    new_w = max(1, int(round(w * (height / h))))
    image = image.resize((new_w, height))

    arr = np.asarray(image, dtype=np.float32) / 255.0
    arr = (arr - 0.5) / 0.5
    arr = np.expand_dims(arr, axis=0)
    return torch.from_numpy(arr)


class OCRDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = preprocess_image(item["image"])
        text = item["text"]
        target = torch.tensor(encode_text(text), dtype=torch.long)
        return image, target, text


def collate_fn(batch):
    images, targets, texts = zip(*batch)
    widths = torch.tensor([img.shape[-1] for img in images], dtype=torch.long)
    max_width = int(widths.max())

    padded_images = []
    for img in images:
        pad_width = max_width - img.shape[-1]
        padded_images.append(F.pad(img, (0, pad_width, 0, 0), value=1.0))

    images = torch.stack(padded_images)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    targets = torch.cat(targets)
    return images, widths, targets, target_lengths, texts

In [ ]:
DEBUG = True
if DEBUG:
    train_samples_run = train_samples[:2500]
    val_samples_run = val_samples[:500]
else:
    train_samples_run = train_samples
    val_samples_run = val_samples

BATCH_SIZE = 32
train_loader = DataLoader(
    OCRDataset(train_samples_run),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    OCRDataset(val_samples_run),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
)

print(len(train_samples_run), len(val_samples_run))

## 6. Model: CRNN + CTC

The idea:

1. CNN reads the crop and extracts visual features.
2. We squeeze feature height to `1`; feature width becomes the time axis `T`.
3. BiLSTM reads the sequence from both directions.
4. A linear layer predicts character probabilities for every time step.
5. CTC loss trains without character-level alignment, so we do not need to know where each letter starts in pixels.

In [ ]:
class CRNN(nn.Module):
    def __init__(self, num_classes, hidden_size=256, dropout=0.1):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),

            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, None)),
        )
        self.rnn = nn.LSTM(
            input_size=256,
            hidden_size=hidden_size,
            num_layers=2,
            bidirectional=True,
            dropout=dropout,
        )
        self.classifier = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, images):
        features = self.cnn(images)      # B, C, 1, W'
        features = features.squeeze(2)   # B, C, W'
        features = features.permute(2, 0, 1)  # T, B, C
        sequence, _ = self.rnn(features)
        logits = self.classifier(sequence)
        return F.log_softmax(logits, dim=-1)

    @staticmethod
    def output_lengths(widths):
        return torch.div(widths, 4, rounding_mode="floor").clamp(min=1)


MODEL_CONFIG = {"hidden_size": 256, "dropout": 0.1}
model = CRNN(num_classes=num_classes, **MODEL_CONFIG).to(device)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

batch = next(iter(train_loader))
images, widths, targets, target_lengths, batch_texts = batch
with torch.no_grad():
    log_probs = model(images.to(device))
print("images:", images.shape)
print("log_probs T,B,C:", log_probs.shape)
print("input lengths:", CRNN.output_lengths(widths)[:5].tolist())

## 7. Metric: CER

CER means character error rate. It is Levenshtein distance between predicted text and target text, divided by the number of target characters. For OCR this is more useful than exact-match accuracy.

In [ ]:
def levenshtein_distance(a, b):
    if len(a) < len(b):
        a, b = b, a
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        current = [i]
        for j, cb in enumerate(b, start=1):
            insert = current[j - 1] + 1
            delete = previous[j] + 1
            replace = previous[j - 1] + (ca != cb)
            current.append(min(insert, delete, replace))
        previous = current
    return previous[-1]


def cer(prediction, target):
    target = normalize_text(target)
    prediction = normalize_text(prediction)
    if len(target) == 0:
        return 0.0 if len(prediction) == 0 else 1.0
    return levenshtein_distance(prediction, target) / len(target)


assert levenshtein_distance("kitten", "sitting") == 3
assert cer("abc", "abc") == 0.0

## 8. Train / validation loop

The first run uses `DEBUG=True`, so training uses only a small subset. Once the pipeline is clear, set `DEBUG=False`, increase `EPOCHS`, and optionally add the `silver` split.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for images, widths, targets, target_lengths, _ in loader:
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)
        input_lengths = model.output_lengths(widths).to(device)

        optimizer.zero_grad(set_to_none=True)
        log_probs = model(images)
        loss = criterion(log_probs, targets, input_lengths, target_lengths)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, max_batches=None):
    model.eval()
    total_loss = 0.0
    total_cer = 0.0
    total_items = 0
    examples = []

    for batch_idx, (images, widths, targets, target_lengths, texts) in enumerate(loader):
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)
        input_lengths = model.output_lengths(widths).to(device)

        log_probs = model(images)
        loss = criterion(log_probs, targets, input_lengths, target_lengths)
        predictions = log_probs.argmax(dim=-1).permute(1, 0).cpu().numpy()

        for pred_indices, target_text in zip(predictions, texts):
            pred_text = ctc_decode(pred_indices)
            item_cer = cer(pred_text, target_text)
            total_cer += item_cer
            total_items += 1
            if len(examples) < 5:
                examples.append((target_text, pred_text, item_cer))

        total_loss += loss.item() * images.size(0)
        if max_batches is not None and batch_idx + 1 >= max_batches:
            break

    denom = total_items if max_batches is not None else len(loader.dataset)
    return {
        "loss": total_loss / denom,
        "cer": total_cer / total_items,
        "examples": examples,
    }

In [ ]:
EPOCHS = 3
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_metrics = evaluate(model, val_loader, criterion)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_cer": val_metrics["cer"],
    }
    history.append(row)
    print(row)

    for truth, pred, item_cer in val_metrics["examples"][:3]:
        print("TARGET:", truth)
        print("PRED:  ", pred)
        print("CER:", round(item_cer, 3))
        print("-" * 80)

In [ ]:
pd.DataFrame(history).plot(x="epoch", y=["train_loss", "val_loss", "val_cer"], marker="o", figsize=(8, 4))
plt.grid(True)
plt.show()

## 9. Save and load the trained OCR model

After training, save both the neural network weights and the OCR metadata. The metadata is not optional: without `idx2char`, `char2idx`, and image preprocessing settings, the weights alone cannot decode predictions back into text.

In [ ]:
CHECKPOINT_PATH = "crnn_rukopys.pt"


def save_checkpoint(path, model, optimizer=None, history=None):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "idx2char": idx2char,
        "char2idx": char2idx,
        "image_height": IMAGE_HEIGHT,
        "model_config": MODEL_CONFIG,
        "normalization": "grayscale resized to IMAGE_HEIGHT, scaled to [-1, 1]",
        "history": history or [],
    }
    if optimizer is not None:
        checkpoint["optimizer_state_dict"] = optimizer.state_dict()
    torch.save(checkpoint, path)
    return path


def load_checkpoint(path, device):
    checkpoint = torch.load(path, map_location=device)
    loaded_idx2char = checkpoint["idx2char"]
    loaded_char2idx = checkpoint["char2idx"]
    loaded_model = CRNN(
        num_classes=len(loaded_idx2char),
        **checkpoint.get("model_config", {"hidden_size": 256, "dropout": 0.1}),
    ).to(device)
    loaded_model.load_state_dict(checkpoint["model_state_dict"])
    loaded_model.eval()
    return loaded_model, loaded_idx2char, loaded_char2idx, checkpoint


save_checkpoint(CHECKPOINT_PATH, model, optimizer=optimizer, history=history)
loaded_model, loaded_idx2char, loaded_char2idx, checkpoint = load_checkpoint(CHECKPOINT_PATH, device)

print("saved:", CHECKPOINT_PATH)
print("classes:", len(loaded_idx2char))
print("history rows:", len(checkpoint.get("history", [])))

## 10. Predict one random validation crop

Early predictions can be empty or repetitive. That is normal for CTC at the beginning. The important signs are falling loss and gradually improving CER. This example uses `loaded_model` to prove the saved checkpoint is usable for inference.

In [ ]:
@torch.no_grad()
def predict_text(model, image):
    model.eval()
    tensor = preprocess_image(image).unsqueeze(0).to(device)
    log_probs = model(tensor)
    pred = log_probs.argmax(dim=-1).squeeze(1).cpu().numpy()
    return ctc_decode(pred)

idx = random.randrange(len(val_samples_run))
item = val_samples_run[idx]
prediction = predict_text(loaded_model, item["image"])

plt.figure(figsize=(10, 2))
plt.imshow(item["image"])
plt.axis("off")
plt.show()

print("target:", item["text"])
print("pred:  ", prediction)
print("CER:", round(cer(prediction, item["text"]), 3))

## 11. From this baseline to a Kaggle submission

This notebook solves the OCR part: `crop -> text`. The full RUKOPYS/Kaggle pipeline also needs `page -> bboxes`. A typical path:

1. Train a layout detector on `regions`: YOLO, RT-DETR, or Faster R-CNN for bbox + `type`.
2. Run it on each test page and sort lines top-to-bottom / left-to-right.
3. Send every detected `handwritten` crop through this OCR model.
4. Format the result according to Kaggle's `sample_submission.csv`.

The next practical step is exporting train regions to YOLO format and training a small detector for `handwritten`, `printed`, `formula`, `table`, `annotation`, `image`, and `graph`.